In [3]:
import pandas as pd
import numpy as np

ratings = pd.read_csv(
    "../data/raw/ratings.csv",
    sep="\t"
)

# Keep users with at least 5 ratings
user_rating_counts = ratings.groupby("userId").size()

eligible_users = user_rating_counts[
    user_rating_counts >= 5
].index

ratings_filtered = ratings[
    ratings["userId"].isin(eligible_users)
].copy()

# Random 80/20 split for each user
train_parts = []
test_parts = []

for user_id, user_data in ratings_filtered.groupby("userId"):
    user_data = user_data.sample(
        frac=1,
        random_state=42
    )

    split_index = int(len(user_data) * 0.8)

    train_parts.append(
        user_data.iloc[:split_index]
    )

    test_parts.append(
        user_data.iloc[split_index:]
    )

train_ratings = pd.concat(
    train_parts,
    ignore_index=True
)

test_ratings = pd.concat(
    test_parts,
    ignore_index=True
)

print("Train ratings:", len(train_ratings))
print("Test ratings:", len(test_ratings))
print("Users:", ratings_filtered["userId"].nunique())

Train ratings: 836087
Test ratings: 212488
Users: 7045


In [4]:
import pandas as pd
import numpy as np

print("Evaluation environment ready.")

Evaluation environment ready.


In [5]:
ratings = pd.read_csv(
    "../data/raw/ratings.csv",
    sep="\t"
)

print("Ratings shape:", ratings.shape)
ratings.head()

Ratings shape: (1048575, 4)


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [6]:
ratings["rating"].describe()

count    1.048575e+06
mean     3.535808e+00
std      1.056276e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

In [7]:
RELEVANCE_THRESHOLD = 4.0

print(
    f"Movies with rating >= {RELEVANCE_THRESHOLD} "
    "will be considered relevant."
)

Movies with rating >= 4.0 will be considered relevant.


In [8]:
# Count how many ratings each user has given
user_rating_counts = ratings.groupby("userId").size()

# Keep users with at least 5 ratings
eligible_users = user_rating_counts[
    user_rating_counts >= 5
].index

ratings_filtered = ratings[
    ratings["userId"].isin(eligible_users)
].copy()

print("Original ratings:", len(ratings))
print("Filtered ratings:", len(ratings_filtered))
print("Eligible users:", ratings_filtered["userId"].nunique())

Original ratings: 1048575
Filtered ratings: 1048575
Eligible users: 7045


In [9]:
ratings_filtered = ratings_filtered.sort_values(
    ["userId", "timestamp"]
).reset_index(drop=True)

ratings_filtered.head()

,userId,movieId,rating,timestamp
0,1,5952,4.0,1147868053
1,1,2012,2.5,1147868068
2,1,2011,2.5,1147868079
3,1,1653,4.0,1147868097
4,1,1250,4.0,1147868414


In [10]:
test_ratings = (
    ratings_filtered
    .groupby("userId")
    .tail(1)
)

print("Test ratings:", len(test_ratings))
print("Test users:", test_ratings["userId"].nunique())

Test ratings: 7045
Test users: 7045


In [11]:
train_ratings = ratings_filtered.drop(
    test_ratings.index
)

print("Train ratings:", len(train_ratings))
print("Test ratings:", len(test_ratings))

print(
    "\nTrain users:",
    train_ratings["userId"].nunique()
)

print(
    "Test users:",
    test_ratings["userId"].nunique()
)

Train ratings: 1041530
Test ratings: 7045

Train users: 7045
Test users: 7045


In [12]:
train_users = set(train_ratings["userId"])
test_users = set(test_ratings["userId"])

overlap = train_users.intersection(test_users)

print("Users in training:", len(train_users))
print("Users in testing:", len(test_users))
print("Users appearing in both:", len(overlap))

Users in training: 7045
Users in testing: 7045
Users appearing in both: 7045


In [13]:
train_pairs = set(
    zip(
        train_ratings["userId"],
        train_ratings["movieId"]
    )
)

test_pairs = set(
    zip(
        test_ratings["userId"],
        test_ratings["movieId"]
    )
)

leakage = train_pairs.intersection(test_pairs)

print("Train/test interaction overlap:", len(leakage))

Train/test interaction overlap: 0


In [14]:
test_ratings["rating"].describe()

count    7045.000000
mean        3.726260
std         1.059377
min         0.500000
25%         3.000000
50%         4.000000
75%         4.500000
max         5.000000
Name: rating, dtype: float64

In [15]:
print(
    "Relevant test ratings:",
    (test_ratings["rating"] >= RELEVANCE_THRESHOLD).sum()
)

print(
    "Total test ratings:",
    len(test_ratings)
)

Relevant test ratings: 4157
Total test ratings: 7045


In [16]:
test_relevant = test_ratings[
    test_ratings["rating"] >= RELEVANCE_THRESHOLD
].copy()

print("Relevant test interactions:", len(test_relevant))
print("Users with relevant test movies:", test_relevant["userId"].nunique())

Relevant test interactions: 4157
Users with relevant test movies: 4157


In [17]:
def precision_at_k(recommended_items, relevant_items, k):
    """
    Calculate Precision@K.

    Precision = relevant recommendations in top K / K
    """

    recommended_top_k = recommended_items[:k]

    relevant_count = sum(
        item in relevant_items
        for item in recommended_top_k
    )

    return relevant_count / k

In [18]:
recommended = [101, 102, 103, 104, 105]
relevant = {101, 103, 200}

print(precision_at_k(recommended, relevant, 5))

0.4


In [19]:
def recall_at_k(recommended_items, relevant_items, k):
    """
    Calculate Recall@K.

    Recall = relevant recommendations in top K
             / total relevant items
    """

    if len(relevant_items) == 0:
        return 0.0

    recommended_top_k = recommended_items[:k]

    relevant_count = sum(
        item in relevant_items
        for item in recommended_top_k
    )

    return relevant_count / len(relevant_items)

In [20]:
print(recall_at_k(recommended, relevant, 5))

0.6666666666666666


In [21]:
import numpy as np


def ndcg_at_k(recommended_items, relevant_items, k):
    """
    Calculate binary NDCG@K.
    """

    recommended_top_k = recommended_items[:k]

    dcg = 0.0

    for position, item in enumerate(
        recommended_top_k,
        start=1
    ):
        if item in relevant_items:
            dcg += 1 / np.log2(position + 1)

    ideal_relevant = min(
        len(relevant_items),
        k
    )

    idcg = sum(
        1 / np.log2(position + 1)
        for position in range(
            1,
            ideal_relevant + 1
        )
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [22]:
print("Precision@5:",
      precision_at_k(recommended, relevant, 5))

print("Recall@5:",
      recall_at_k(recommended, relevant, 5))

print("NDCG@5:",
      ndcg_at_k(recommended, relevant, 5))

Precision@5: 0.4
Recall@5: 0.6666666666666666
NDCG@5: 0.7039180890341347


In [23]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import linear_kernel

movie_features = pd.read_csv(
    "../data/processed/movie_features.csv"
)

print("Movie features loaded:", movie_features.shape)

Movie features loaded: (62423, 6)


In [24]:
# Select one user from the test set
sample_user = test_relevant["userId"].iloc[0]

user_history = train_ratings[
    train_ratings["userId"] == sample_user
].sort_values(
    "rating",
    ascending=False
)

print("Sample user:", sample_user)
print("Number of training ratings:", len(user_history))

user_history.head(10)

Sample user: 1
Number of training ratings: 69


,userId,movieId,rating,timestamp
29,1,6016,5.0,1147869090
13,1,3949,5.0,1147868678
18,1,307,5.0,1147868828
19,1,1237,5.0,1147868839
12,1,6711,5.0,1147868622
53,1,2632,5.0,1147878248
57,1,5767,5.0,1147878729
58,1,665,5.0,1147878820
61,1,8327,5.0,1147879375
68,1,296,5.0,1147880044


In [25]:
user_liked_movies = user_history[
    user_history["rating"] >= RELEVANCE_THRESHOLD
]

print(
    "Liked movies in training:",
    len(user_liked_movies)
)

user_liked_movies.head(10)

Liked movies in training: 38


,userId,movieId,rating,timestamp
29,1,6016,5.0,1147869090
13,1,3949,5.0,1147868678
18,1,307,5.0,1147868828
19,1,1237,5.0,1147868839
12,1,6711,5.0,1147868622
53,1,2632,5.0,1147878248
57,1,5767,5.0,1147878729
58,1,665,5.0,1147878820
61,1,8327,5.0,1147879375
68,1,296,5.0,1147880044


In [26]:
liked_movie_indices = []

for movie_id in user_liked_movies["movieId"]:
    
    matches = movie_features.index[
        movie_features["movieId"] == movie_id
    ]
    
    if len(matches) > 0:
        liked_movie_indices.append(matches[0])

print(
    "Liked movies found in content model:",
    len(liked_movie_indices)
)

Liked movies found in content model: 38


In [27]:
from scipy import sparse
from sklearn.metrics.pairwise import linear_kernel

# Load the saved weighted TF-IDF matrix
weighted_tfidf_matrix = sparse.load_npz(
    "../models/weighted_tfidf_matrix.npz"
)

print(
    "Weighted TF-IDF matrix loaded:",
    weighted_tfidf_matrix.shape
)

Weighted TF-IDF matrix loaded: (62423, 35768)


In [28]:
user_similarity_scores = np.zeros(
    len(movie_features)
)

for movie_idx in liked_movie_indices:

    similarity_scores = linear_kernel(
        weighted_tfidf_matrix[movie_idx],
        weighted_tfidf_matrix
    ).flatten()

    user_similarity_scores += similarity_scores

if len(liked_movie_indices) > 0:
    user_similarity_scores /= len(
        liked_movie_indices
    )

print(
    "Generated scores for",
    len(user_similarity_scores),
    "movies."
)

Generated scores for 62423 movies.


In [29]:
# Get all movies the user has already rated
seen_movie_ids = set(
    train_ratings[
        train_ratings["userId"] == sample_user
    ]["movieId"]
)

print(
    "Movies already seen by user:",
    len(seen_movie_ids)
)

Movies already seen by user: 69


In [30]:
seen_movie_indices = set()

for movie_id in seen_movie_ids:

    matches = movie_features.index[
        movie_features["movieId"] == movie_id
    ]

    if len(matches) > 0:
        seen_movie_indices.add(matches[0])

print(
    "Seen movies found in content model:",
    len(seen_movie_indices)
)

Seen movies found in content model: 69


In [31]:
candidate_scores = user_similarity_scores.copy()

for movie_idx in seen_movie_indices:
    candidate_scores[movie_idx] = -np.inf

print(
    "Candidate movies:",
    np.isfinite(candidate_scores).sum()
)

Candidate movies: 62354


In [32]:
top_k = 10

top_indices = np.argsort(
    candidate_scores
)[-top_k:][::-1]

user_recommendations = movie_features.iloc[
    top_indices
][
    ["movieId", "title", "genres"]
].copy()

user_recommendations["score"] = (
    candidate_scores[top_indices]
)

user_recommendations

,movieId,title,genres,score
11117,48394,"Pan's Labyrinth (Laberinto del fauno, El) (2006)",Drama|Fantasy|Thriller,0.087772
121,123,Chungking Express (Chung Hing sam lam) (1994),Drama|Mystery|Romance,0.079852
2336,2427,"Thin Red Line, The (1998)",Action|Drama|War,0.076308
12665,61937,Sawdust and Tinsel (Gycklarnas afton) (1953),Drama,0.075123
6942,7067,Juliet of the Spirits (Giulietta degli spiriti...,Comedy|Drama|Fantasy|Romance,0.074843
11075,48043,"Fountain, The (2006)",Drama|Fantasy|Romance,0.073740
6846,6971,Europa (Zentropa) (1991),Drama|Thriller,0.073553
3405,3503,Solaris (Solyaris) (1972),Drama|Mystery|Sci-Fi,0.072330
10405,40738,Summer Interlude (Sommarlek) (1951),Drama|Romance,0.071585
3457,3556,"Virgin Suicides, The (1999)",Drama|Romance,0.070396


In [33]:
user_test_relevant = test_relevant[
    test_relevant["userId"] == sample_user
]

test_movie_ids = set(
    user_test_relevant["movieId"]
)

print("User:", sample_user)
print("Relevant test movies:", test_movie_ids)

User: 1
Relevant test movies: {7361}


In [34]:
recommended_movie_ids = (
    user_recommendations["movieId"]
    .tolist()
)

print(
    "Recommended movie IDs:",
    recommended_movie_ids
)

Recommended movie IDs: [48394, 123, 2427, 61937, 7067, 48043, 6971, 3503, 40738, 3556]


In [35]:
precision = precision_at_k(
    recommended_movie_ids,
    test_movie_ids,
    10
)

print("Precision@10:", precision)

Precision@10: 0.0


In [36]:
recall = recall_at_k(
    recommended_movie_ids,
    test_movie_ids,
    10
)

ndcg = ndcg_at_k(
    recommended_movie_ids,
    test_movie_ids,
    10
)

print("Recall@10:", recall)
print("NDCG@10:", ndcg)

Recall@10: 0.0
NDCG@10: 0.0


In [37]:
# Use a manageable sample for the first full evaluation
EVAL_USERS = 500

evaluation_users = (
    test_relevant["userId"]
    .drop_duplicates()
    .sample(
        n=min(EVAL_USERS, test_relevant["userId"].nunique()),
        random_state=42
    )
)

print("Evaluation users:", len(evaluation_users))

Evaluation users: 500


In [38]:
# Ground-truth relevant movies for each evaluation user
test_relevant_by_user = (
    test_relevant[
        test_relevant["userId"].isin(evaluation_users)
    ]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print(
    "Users with test targets:",
    len(test_relevant_by_user)
)

Users with test targets: 500


In [39]:
def generate_user_recommendations(
    user_id,
    train_ratings,
    movie_features,
    weighted_tfidf_matrix,
    n=10
):
    # User's training history
    user_history = train_ratings[
        train_ratings["userId"] == user_id
    ]

    # Movies the user liked
    liked_movies = user_history[
        user_history["rating"] >= RELEVANCE_THRESHOLD
    ]

    if len(liked_movies) == 0:
        return []

    # Convert movie IDs to movie-feature indices
    liked_movie_indices = []

    for movie_id in liked_movies["movieId"]:
        matches = movie_features.index[
            movie_features["movieId"] == movie_id
        ]

        if len(matches) > 0:
            liked_movie_indices.append(matches[0])

    if len(liked_movie_indices) == 0:
        return []

    # Build user preference scores
    user_scores = np.zeros(
        len(movie_features)
    )

    for movie_idx in liked_movie_indices:

        similarity_scores = linear_kernel(
            weighted_tfidf_matrix[movie_idx],
            weighted_tfidf_matrix
        ).flatten()

        user_scores += similarity_scores

    user_scores /= len(liked_movie_indices)

    # Remove movies already seen
    seen_movie_ids = set(
        user_history["movieId"]
    )

    for movie_id in seen_movie_ids:

        matches = movie_features.index[
            movie_features["movieId"] == movie_id
        ]

        if len(matches) > 0:
            user_scores[matches[0]] = -np.inf

    # Top-N recommendations
    top_indices = np.argsort(
        user_scores
    )[-n:][::-1]

    return movie_features.iloc[
        top_indices
    ]["movieId"].tolist()

In [40]:
test_user = evaluation_users.iloc[0]

recommendations = generate_user_recommendations(
    test_user,
    train_ratings,
    movie_features,
    weighted_tfidf_matrix,
    n=10
)

print("User:", test_user)
print("Recommendations:", recommendations)
print(
    "Relevant test movies:",
    test_relevant_by_user.get(test_user, set())
)

User: 4172
Recommendations: [1080, 3033, 143902, 2788, 6104, 158270, 166619, 181365, 135743, 1270]
Relevant test movies: {5679}


In [41]:
K = 10

evaluation_results = []

for i, user_id in enumerate(evaluation_users, start=1):

    recommendations = generate_user_recommendations(
        user_id,
        train_ratings,
        movie_features,
        weighted_tfidf_matrix,
        n=K
    )

    relevant_movies = test_relevant_by_user.get(
        user_id,
        set()
    )

    if len(recommendations) == 0 or len(relevant_movies) == 0:
        continue

    precision = precision_at_k(
        recommendations,
        relevant_movies,
        K
    )

    recall = recall_at_k(
        recommendations,
        relevant_movies,
        K
    )

    ndcg = ndcg_at_k(
        recommendations,
        relevant_movies,
        K
    )

    evaluation_results.append({
        "userId": user_id,
        "precision@10": precision,
        "recall@10": recall,
        "ndcg@10": ndcg
    })

    if i % 50 == 0:
        print(f"Processed {i}/{len(evaluation_users)} users")

Processed 50/500 users
Processed 100/500 users
Processed 150/500 users
Processed 200/500 users
Processed 250/500 users
Processed 300/500 users
Processed 350/500 users
Processed 400/500 users
Processed 450/500 users
Processed 500/500 users


In [42]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

print(
    "Successfully evaluated users:",
    len(evaluation_df)
)

evaluation_df.head()

Successfully evaluated users: 500


,userId,precision@10,recall@10,ndcg@10
0,4172,0.0,0.0,0.0
1,497,0.0,0.0,0.0
2,4060,0.0,0.0,0.0
3,4591,0.0,0.0,0.0
4,766,0.0,0.0,0.0


In [43]:
mean_precision = evaluation_df["precision@10"].mean()
mean_recall = evaluation_df["recall@10"].mean()
mean_ndcg = evaluation_df["ndcg@10"].mean()

print(f"Mean Precision@10: {mean_precision:.4f}")
print(f"Mean Recall@10:    {mean_recall:.4f}")
print(f"Mean NDCG@10:      {mean_ndcg:.4f}")

Mean Precision@10: 0.0036
Mean Recall@10:    0.0360
Mean NDCG@10:      0.0198


In [44]:
evaluation_df[
    ["precision@10", "recall@10", "ndcg@10"]
].describe()

,precision@10,recall@10,ndcg@10
count,500.000000,500.000000,500.000000
mean,0.003600,0.036000,0.019789
std,0.018648,0.186477,0.116346
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,0.100000,1.000000,1.000000


In [45]:
mean_precision = evaluation_df["precision@10"].mean()
mean_recall = evaluation_df["recall@10"].mean()
mean_ndcg = evaluation_df["ndcg@10"].mean()

print(f"Mean Precision@10: {mean_precision:.4f}")
print(f"Mean Recall@10:    {mean_recall:.4f}")
print(f"Mean NDCG@10:      {mean_ndcg:.4f}")

Mean Precision@10: 0.0036
Mean Recall@10:    0.0360
Mean NDCG@10:      0.0198


In [46]:
evaluation_df[
    ["precision@10", "recall@10", "ndcg@10"]
].describe()

,precision@10,recall@10,ndcg@10
count,500.000000,500.000000,500.000000
mean,0.003600,0.036000,0.019789
std,0.018648,0.186477,0.116346
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,0.100000,1.000000,1.000000


In [47]:
import numpy as np


def normalize_scores(scores):
    """
    Normalize scores to the range [0, 1].
    """
    scores = np.asarray(scores, dtype=float)

    min_score = np.min(scores)
    max_score = np.max(scores)

    if max_score == min_score:
        return np.zeros_like(scores)

    return (
        (scores - min_score)
        / (max_score - min_score)
    )

In [48]:
CONTENT_WEIGHT = 0.5
COLLAB_WEIGHT = 0.5

print(
    "Content weight:",
    CONTENT_WEIGHT
)

print(
    "Collaborative weight:",
    COLLAB_WEIGHT
)

Content weight: 0.5
Collaborative weight: 0.5


In [49]:
def generate_hybrid_recommendations(
    user_id,
    train_ratings,
    movie_features,
    weighted_tfidf_matrix,
    predicted_ratings,
    train_user_id_to_index,
    train_movie_id_to_index,
    index_to_train_movie_id,
    n=10,
    content_weight=0.5
):
    """
    Generate personalized recommendations by combining
    content-based and collaborative filtering scores.
    """

    if user_id not in train_user_id_to_index:
        return []

    # --------------------------------------------------
    # 1. CONTENT-BASED SCORES
    # --------------------------------------------------

    user_history = train_ratings[
        train_ratings["userId"] == user_id
    ]

    liked_movies = user_history[
        user_history["rating"] >= RELEVANCE_THRESHOLD
    ]

    liked_movie_indices = []

    for movie_id in liked_movies["movieId"]:
        matches = movie_features.index[
            movie_features["movieId"] == movie_id
        ]

        if len(matches) > 0:
            liked_movie_indices.append(matches[0])

    if len(liked_movie_indices) == 0:
        return []

    content_scores = np.zeros(
        len(movie_features)
    )

    for movie_idx in liked_movie_indices:

        similarity_scores = linear_kernel(
            weighted_tfidf_matrix[movie_idx],
            weighted_tfidf_matrix
        ).flatten()

        content_scores += similarity_scores

    content_scores /= len(
        liked_movie_indices
    )

    # --------------------------------------------------
    # 2. COLLABORATIVE FILTERING SCORES
    # --------------------------------------------------

    user_idx = train_user_id_to_index[user_id]

    svd_scores = predicted_ratings[
        user_idx
    ].copy()

    # --------------------------------------------------
    # 3. MAP SVD SCORES TO MOVIE IDs
    # --------------------------------------------------

    collaborative_scores = np.zeros(
        len(movie_features)
    )

    for movie_idx, movie_id in (
        index_to_train_movie_id.items()
    ):

        movie_matches = movie_features.index[
            movie_features["movieId"] == movie_id
        ]

        if len(movie_matches) > 0:
            movie_feature_idx = movie_matches[0]

            collaborative_scores[
                movie_feature_idx
            ] = svd_scores[movie_idx]

    # --------------------------------------------------
    # 4. NORMALIZE BOTH SCORE TYPES
    # --------------------------------------------------

    normalized_content = normalize_scores(
        content_scores
    )

    normalized_collaborative = normalize_scores(
        collaborative_scores
    )

    # --------------------------------------------------
    # 5. HYBRID SCORE
    # --------------------------------------------------

    hybrid_scores = (
        content_weight * normalized_content
        +
        (1 - content_weight)
        * normalized_collaborative
    )

    # --------------------------------------------------
    # 6. REMOVE MOVIES ALREADY SEEN
    # --------------------------------------------------

    seen_movie_ids = set(
        user_history["movieId"]
    )

    for movie_id in seen_movie_ids:

        matches = movie_features.index[
            movie_features["movieId"] == movie_id
        ]

        if len(matches) > 0:
            hybrid_scores[
                matches[0]
            ] = -np.inf

    # --------------------------------------------------
    # 7. GET TOP-K
    # --------------------------------------------------

    top_indices = np.argsort(
        hybrid_scores
    )[-n:][::-1]

    recommendations = movie_features.iloc[
        top_indices
    ][
        ["movieId", "title", "genres"]
    ].copy()

    recommendations["hybrid_score"] = (
        hybrid_scores[top_indices]
    )

    return recommendations

In [50]:
train_movie_ids = train_ratings["movieId"].unique()

train_movie_id_to_index = {
    movie_id: idx
    for idx, movie_id in enumerate(train_movie_ids)
}

index_to_train_movie_id = {
    idx: movie_id
    for movie_id, idx in train_movie_id_to_index.items()
}

print(
    "Training movies:",
    len(train_movie_id_to_index)
)

Training movies: 22187


In [51]:
import numpy as np

def normalize_scores(scores):
    """
    Normalize scores to the range [0, 1].
    """
    scores = np.asarray(scores, dtype=float)

    min_score = np.min(scores)
    max_score = np.max(scores)

    if max_score == min_score:
        return np.zeros_like(scores)

    return (
        (scores - min_score)
        / (max_score - min_score)
    )

In [52]:
import numpy as np


def precision_at_k(
    recommended_items,
    relevant_items,
    k
):
    recommended_top_k = recommended_items[:k]

    relevant_count = sum(
        item in relevant_items
        for item in recommended_top_k
    )

    return relevant_count / k


def recall_at_k(
    recommended_items,
    relevant_items,
    k
):
    if len(relevant_items) == 0:
        return 0.0

    recommended_top_k = recommended_items[:k]

    relevant_count = sum(
        item in relevant_items
        for item in recommended_top_k
    )

    return (
        relevant_count
        / len(relevant_items)
    )


def ndcg_at_k(
    recommended_items,
    relevant_items,
    k
):
    recommended_top_k = recommended_items[:k]

    dcg = 0.0

    for position, item in enumerate(
        recommended_top_k,
        start=1
    ):
        if item in relevant_items:
            dcg += 1 / np.log2(
                position + 1
            )

    ideal_relevant = min(
        len(relevant_items),
        k
    )

    idcg = sum(
        1 / np.log2(position + 1)
        for position in range(
            1,
            ideal_relevant + 1
        )
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [53]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

train_user_ids = train_ratings["userId"].unique()

train_user_id_to_index = {
    user_id: idx
    for idx, user_id in enumerate(train_user_ids)
}

row_indices = train_ratings["userId"].map(
    train_user_id_to_index
).values

col_indices = train_ratings["movieId"].map(
    train_movie_id_to_index
).values

rating_values = train_ratings["rating"].values

train_user_item_matrix = csr_matrix(
    (
        rating_values,
        (row_indices, col_indices)
    ),
    shape=(
        len(train_user_id_to_index),
        len(train_movie_id_to_index)
    )
)

svd_train = TruncatedSVD(
    n_components=50,
    random_state=42
)

user_latent_train = svd_train.fit_transform(
    train_user_item_matrix
)

predicted_ratings = (
    user_latent_train
    @ svd_train.components_
)

print(
    "User latent matrix:",
    user_latent_train.shape
)

print(
    "Predicted ratings:",
    predicted_ratings.shape
)

User latent matrix: (7045, 50)
Predicted ratings: (7045, 22187)


In [54]:
test_ratings_eval = test_ratings[
    test_ratings["movieId"].isin(
        train_movie_id_to_index
    )
].copy()

RELEVANCE_THRESHOLD = 4.0

test_relevant_eval = test_ratings_eval[
    test_ratings_eval["rating"] >= RELEVANCE_THRESHOLD
].copy()

evaluation_users_svd = (
    test_relevant_eval["userId"]
    .drop_duplicates()
    .tolist()
)

test_relevant_by_user_svd = (
    test_relevant_eval
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print(
    "Evaluation users:",
    len(evaluation_users_svd)
)

Evaluation users: 4136


In [55]:
import pandas as pd

movie_features = pd.read_csv(
    "../data/processed/movie_features.csv"
)

print("Movie features shape:", movie_features.shape)
print(movie_features.columns.tolist())

Movie features shape: (62423, 6)
['movieId', 'title', 'genres', 'tag', 'content', 'weighted_content']


In [56]:
from scipy import sparse
from sklearn.metrics.pairwise import linear_kernel

weighted_tfidf_matrix = sparse.load_npz(
    "../models/weighted_tfidf_matrix.npz"
)

print(
    "Weighted TF-IDF matrix shape:",
    weighted_tfidf_matrix.shape
)

Weighted TF-IDF matrix shape: (62423, 35768)


In [57]:
K = 10

hybrid_evaluation_results = []

for i, user_id in enumerate(
    evaluation_users_svd,
    start=1
):
    recommendations = generate_hybrid_recommendations(
        user_id=user_id,
        train_ratings=train_ratings,
        movie_features=movie_features,
        weighted_tfidf_matrix=weighted_tfidf_matrix,
        predicted_ratings=predicted_ratings,
        train_user_id_to_index=train_user_id_to_index,
        train_movie_id_to_index=train_movie_id_to_index,
        index_to_train_movie_id=index_to_train_movie_id,
        n=K,
        content_weight=0.5
    )

    recommended_movies = recommendations[
        "movieId"
    ].tolist()

    relevant_movies = test_relevant_by_user_svd.get(
        user_id,
        set()
    )

    if not recommended_movies or not relevant_movies:
        continue

    hybrid_evaluation_results.append({
        "userId": user_id,
        "precision@10": precision_at_k(
            recommended_movies,
            relevant_movies,
            K
        ),
        "recall@10": recall_at_k(
            recommended_movies,
            relevant_movies,
            K
        ),
        "ndcg@10": ndcg_at_k(
            recommended_movies,
            relevant_movies,
            K
        )
    })

    if i % 100 == 0:
        print(
            f"Processed {i}/{len(evaluation_users_svd)} users"
        )

KeyboardInterrupt: 

In [58]:
from scipy import sparse
from sklearn.metrics.pairwise import linear_kernel

def generate_content_profile_scores(
    user_id,
    train_ratings,
    movie_features,
    weighted_tfidf_matrix
):
    user_history = train_ratings[
        train_ratings["userId"] == user_id
    ]

    liked_movies = user_history[
        user_history["rating"] >= RELEVANCE_THRESHOLD
    ]

    liked_movie_indices = []

    for movie_id in liked_movies["movieId"]:
        matches = movie_features.index[
            movie_features["movieId"] == movie_id
        ]

        if len(matches) > 0:
            liked_movie_indices.append(matches[0])

    if len(liked_movie_indices) == 0:
        return None, user_history

    # Create one profile representing the user's liked movies
    user_profile = sparse.csr_matrix(
        weighted_tfidf_matrix[
            liked_movie_indices
        ].mean(axis=0)
    )

    # Compare the user profile against every movie
    content_scores = linear_kernel(
        user_profile,
        weighted_tfidf_matrix
    ).flatten()

    return content_scores, user_history

In [61]:
test_user = evaluation_users_svd[0]
content_scores, user_history = generate_content_profile_scores(
    test_user,
    train_ratings,
    movie_features,
    weighted_tfidf_matrix
)

print("User:", test_user)
print("Scores generated:", content_scores is not None)
print("Number of movie scores:", len(content_scores))

User: 1
Scores generated: True
Number of movie scores: 62423


In [62]:
movie_feature_to_train_index = np.full(
    len(movie_features),
    -1,
    dtype=int
)

for feature_idx, movie_id in enumerate(
    movie_features["movieId"]
):
    if movie_id in train_movie_id_to_index:
        movie_feature_to_train_index[feature_idx] = (
            train_movie_id_to_index[movie_id]
        )

print(
    "Movies mapped:",
    np.sum(movie_feature_to_train_index >= 0)
)
print(
    "Total movies:",
    len(movie_features)
)

Movies mapped: 22187
Total movies: 62423


In [63]:
def generate_hybrid_recommendations_fast(
    user_id,
    train_ratings,
    movie_features,
    weighted_tfidf_matrix,
    predicted_ratings,
    train_user_id_to_index,
    movie_feature_to_train_index,
    n=10,
    content_weight=0.5
):
    if user_id not in train_user_id_to_index:
        return []

    # --------------------------------
    # 1. CONTENT SCORE
    # --------------------------------
    content_scores, user_history = (
        generate_content_profile_scores(
            user_id,
            train_ratings,
            movie_features,
            weighted_tfidf_matrix
        )
    )

    if content_scores is None:
        return []

    # --------------------------------
    # 2. COLLABORATIVE SCORE
    # --------------------------------
    user_idx = train_user_id_to_index[user_id]

    svd_scores = predicted_ratings[user_idx]

    collaborative_scores = np.zeros(
        len(movie_features)
    )

    valid_mask = (
        movie_feature_to_train_index >= 0
    )

    collaborative_scores[valid_mask] = (
        svd_scores[
            movie_feature_to_train_index[valid_mask]
        ]
    )

    # --------------------------------
    # 3. NORMALIZE
    # --------------------------------
    normalized_content = normalize_scores(
        content_scores
    )

    normalized_collaborative = normalize_scores(
        collaborative_scores
    )

    # --------------------------------
    # 4. HYBRID SCORE
    # --------------------------------
    hybrid_scores = (
        content_weight * normalized_content
        + (1 - content_weight)
        * normalized_collaborative
    )

    # --------------------------------
    # 5. REMOVE ALREADY-SEEN MOVIES
    # --------------------------------
    seen_movie_ids = set(
        user_history["movieId"]
    )

    seen_mask = movie_features["movieId"].isin(
        seen_movie_ids
    )

    hybrid_scores[seen_mask.values] = -np.inf

    # --------------------------------
    # 6. TOP-N
    # --------------------------------
    top_indices = np.argsort(
        hybrid_scores
    )[-n:][::-1]

    recommendations = movie_features.iloc[
        top_indices
    ][
        ["movieId", "title", "genres"]
    ].copy()

    recommendations["hybrid_score"] = (
        hybrid_scores[top_indices]
    )

    return recommendations

In [64]:
test_hybrid = generate_hybrid_recommendations_fast(
    test_user,
    train_ratings,
    movie_features,
    weighted_tfidf_matrix,
    predicted_ratings,
    train_user_id_to_index,
    movie_feature_to_train_index,
    n=10,
    content_weight=0.5
)

test_hybrid

,movieId,title,genres,hybrid_score
7028,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,0.672467
4887,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,0.641682
11117,48394,"Pan's Labyrinth (Laberinto del fauno, El) (2006)",Drama|Fantasy|Thriller,0.627738
7236,7361,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,0.622239
762,778,Trainspotting (1996),Comedy|Crime|Drama,0.579264
121,123,Chungking Express (Chung Hing sam lam) (1994),Drama|Mystery|Romance,0.571861
4742,4848,Mulholland Drive (2001),Crime|Drama|Film-Noir|Mystery|Thriller,0.561183
4772,4878,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,0.560058
3405,3503,Solaris (Solyaris) (1972),Drama|Mystery|Sci-Fi,0.556589
3457,3556,"Virgin Suicides, The (1999)",Drama|Romance,0.551637


In [65]:
EVAL_USERS_HYBRID = 500

evaluation_users_hybrid = (
    pd.Series(evaluation_users_svd)
    .sample(
        n=min(
            EVAL_USERS_HYBRID,
            len(evaluation_users_svd)
        ),
        random_state=42
    )
    .tolist()
)

print(
    "Hybrid evaluation users:",
    len(evaluation_users_hybrid)
)

Hybrid evaluation users: 500


In [66]:
hybrid_evaluation_results = []

for i, user_id in enumerate(
    evaluation_users_hybrid,
    start=1
):
    recommendations = generate_hybrid_recommendations_fast(
        user_id,
        train_ratings,
        movie_features,
        weighted_tfidf_matrix,
        predicted_ratings,
        train_user_id_to_index,
        movie_feature_to_train_index,
        n=10,
        content_weight=0.5
    )

    recommended_items = (
        recommendations["movieId"].tolist()
        if isinstance(recommendations, pd.DataFrame)
        else []
    )

    relevant_items = test_relevant_by_user_svd.get(
        user_id,
        set()
    )

    if len(recommended_items) == 0:
        continue

    hybrid_evaluation_results.append({
        "userId": user_id,
        "precision@10": precision_at_k(
            recommended_items,
            relevant_items,
            10
        ),
        "recall@10": recall_at_k(
            recommended_items,
            relevant_items,
            10
        ),
        "ndcg@10": ndcg_at_k(
            recommended_items,
            relevant_items,
            10
        )
    })

    if i % 50 == 0:
        print(f"Processed {i}/{len(evaluation_users_hybrid)} users")

Processed 50/500 users
Processed 100/500 users
Processed 150/500 users
Processed 200/500 users
Processed 250/500 users
Processed 300/500 users
Processed 350/500 users
Processed 400/500 users
Processed 450/500 users
Processed 500/500 users


In [67]:
hybrid_evaluation_df = pd.DataFrame(
    hybrid_evaluation_results
)

print(
    "Successfully evaluated users:",
    len(hybrid_evaluation_df)
)

hybrid_evaluation_df.head()

Successfully evaluated users: 500


,userId,precision@10,recall@10,ndcg@10
0,4987,0.0,0.0,0.0
1,2041,0.0,0.0,0.0
2,4359,0.0,0.0,0.0
3,2951,0.0,0.0,0.0
4,1847,0.0,0.0,0.0


In [68]:
mean_hybrid_precision = (
    hybrid_evaluation_df["precision@10"].mean()
)

mean_hybrid_recall = (
    hybrid_evaluation_df["recall@10"].mean()
)

mean_hybrid_ndcg = (
    hybrid_evaluation_df["ndcg@10"].mean()
)

print(
    f"Hybrid Mean Precision@10: "
    f"{mean_hybrid_precision:.4f}"
)

print(
    f"Hybrid Mean Recall@10:    "
    f"{mean_hybrid_recall:.4f}"
)

print(
    f"Hybrid Mean NDCG@10:      "
    f"{mean_hybrid_ndcg:.4f}"
)

Hybrid Mean Precision@10: 0.0080
Hybrid Mean Recall@10:    0.0800
Hybrid Mean NDCG@10:      0.0431


In [71]:
%whos

Variable                               Type            Data/Info
----------------------------------------------------------------
COLLAB_WEIGHT                          float           0.5
CONTENT_WEIGHT                         float           0.5
EVAL_USERS                             int             500
EVAL_USERS_HYBRID                      int             500
K                                      int             10
RELEVANCE_THRESHOLD                    float           4.0
TruncatedSVD                           type            <class 'sklearn.decomposi<...>ncated_svd.TruncatedSVD'>
candidate_scores                       ndarray         62423: 62423 elems, type `float64`, 499384 bytes (487.6796875 kb)
col_indices                            ndarray         1041530: 1041530 elems, type `int64`, 8332240 bytes (7.9462432861328125 Mb)
content_scores                         ndarray         62423: 62423 elems, type `float64`, 499384 bytes (487.6796875 kb)
csr_matrix                       

In [72]:
def evaluate_hybrid_weight(content_weight, evaluation_users):
    
    results = []

    for i, user_id in enumerate(
        evaluation_users,
        start=1
    ):
        recommendations = generate_hybrid_recommendations_fast(
            user_id,
            train_ratings,
            movie_features,
            weighted_tfidf_matrix,
            predicted_ratings,
            train_user_id_to_index,
            movie_feature_to_train_index,
            n=10,
            content_weight=content_weight
        )

        recommended_items = (
            recommendations["movieId"].tolist()
            if isinstance(recommendations, pd.DataFrame)
            else []
        )

        relevant_items = test_relevant_by_user_svd.get(
            user_id,
            set()
        )

        if len(recommended_items) == 0:
            continue

        results.append({
            "precision@10": precision_at_k(
                recommended_items,
                relevant_items,
                10
            ),
            "recall@10": recall_at_k(
                recommended_items,
                relevant_items,
                10
            ),
            "ndcg@10": ndcg_at_k(
                recommended_items,
                relevant_items,
                10
            )
        })

    results_df = pd.DataFrame(results)

    return {
        "content_weight": content_weight,
        "collaborative_weight": 1 - content_weight,
        "precision@10": results_df["precision@10"].mean(),
        "recall@10": results_df["recall@10"].mean(),
        "ndcg@10": results_df["ndcg@10"].mean()
    }

In [73]:
weight_results = []

for weight in [0.25, 0.50, 0.75]:
    
    print(f"Evaluating content weight: {weight}")
    
    result = evaluate_hybrid_weight(
        weight,
        evaluation_users_hybrid
    )
    
    weight_results.append(result)

weight_results

Evaluating content weight: 0.25
Evaluating content weight: 0.5
Evaluating content weight: 0.75


[{'content_weight': 0.25,
  'collaborative_weight': 0.75,
  'precision@10': np.float64(0.011400000000000002),
  'recall@10': np.float64(0.114),
  'ndcg@10': np.float64(0.057312177778659074)},
 {'content_weight': 0.5,
  'collaborative_weight': 0.5,
  'precision@10': np.float64(0.008000000000000002),
  'recall@10': np.float64(0.08),
  'ndcg@10': np.float64(0.0431010921523859)},
 {'content_weight': 0.75,
  'collaborative_weight': 0.25,
  'precision@10': np.float64(0.0048000000000000004),
  'recall@10': np.float64(0.048),
  'ndcg@10': np.float64(0.025953667569398133)}]

In [74]:
weight_comparison = pd.DataFrame(weight_results)

weight_comparison

,content_weight,collaborative_weight,precision@10,recall@10,ndcg@10
0,0.25,0.75,0.0114,0.114,0.057312
1,0.50,0.50,0.0080,0.080,0.043101
2,0.75,0.25,0.0048,0.048,0.025954


In [75]:
best_weight = weight_comparison.loc[
    weight_comparison["ndcg@10"].idxmax()
]

best_weight

content_weight          0.250000
collaborative_weight    0.750000
precision@10            0.011400
recall@10               0.114000
ndcg@10                 0.057312
Name: 0, dtype: float64

In [76]:
print("Content evaluation users:", len(evaluation_users))
print("SVD evaluation users:", len(evaluation_users_svd))
print("Hybrid evaluation users:", len(evaluation_users_hybrid))

Content evaluation users: 500
SVD evaluation users: 4136
Hybrid evaluation users: 500


In [77]:
common_evaluation_users = evaluation_users_hybrid

print(
    "Common evaluation users:",
    len(common_evaluation_users)
)

Common evaluation users: 500


In [78]:
common_users_with_ground_truth = [
    user_id
    for user_id in common_evaluation_users
    if user_id in test_relevant_by_user_svd
]

print(
    "Users with test ground truth:",
    len(common_users_with_ground_truth)
)

Users with test ground truth: 500


In [79]:
content_common_results = []

for i, user_id in enumerate(
    common_evaluation_users,
    start=1
):
    recommendations = generate_user_recommendations(
        user_id,
        train_ratings,
        movie_features,
        weighted_tfidf_matrix,
        n=10
    )

    relevant_items = test_relevant_by_user_svd.get(
        user_id,
        set()
    )

    if len(recommendations) == 0:
        continue

    content_common_results.append({
        "userId": user_id,
        "precision@10": precision_at_k(
            recommendations,
            relevant_items,
            10
        ),
        "recall@10": recall_at_k(
            recommendations,
            relevant_items,
            10
        ),
        "ndcg@10": ndcg_at_k(
            recommendations,
            relevant_items,
            10
        )
    })

    if i % 100 == 0:
        print(
            f"Processed {i}/"
            f"{len(common_evaluation_users)} users"
        )

Processed 100/500 users
Processed 200/500 users
Processed 300/500 users
Processed 400/500 users
Processed 500/500 users


In [80]:
content_common_df = pd.DataFrame(
    content_common_results
)

mean_content_precision_common = (
    content_common_df["precision@10"].mean()
)

mean_content_recall_common = (
    content_common_df["recall@10"].mean()
)

mean_content_ndcg_common = (
    content_common_df["ndcg@10"].mean()
)

print(
    f"Content Precision@10: "
    f"{mean_content_precision_common:.4f}"
)

print(
    f"Content Recall@10:    "
    f"{mean_content_recall_common:.4f}"
)

print(
    f"Content NDCG@10:      "
    f"{mean_content_ndcg_common:.4f}"
)

Content Precision@10: 0.0036
Content Recall@10:    0.0360
Content NDCG@10:      0.0188


In [81]:
evaluation_df[[
    "precision@10",
    "recall@10",
    "ndcg@10"
]].mean()

precision@10    0.003600
recall@10       0.036000
ndcg@10         0.019789
dtype: float64

In [85]:
def generate_svd_recommendations(user_id, n=10):
    if user_id not in train_user_id_to_index:
        return []

    user_idx = train_user_id_to_index[user_id]

    user_scores = predicted_ratings[
        user_idx
    ].copy()

    # Remove movies the user already rated
    seen_movie_ids = set(
        train_ratings[
            train_ratings["userId"] == user_id
        ]["movieId"]
    )

    for movie_id in seen_movie_ids:
        if movie_id in train_movie_id_to_index:
            movie_idx = train_movie_id_to_index[
                movie_id
            ]

            user_scores[movie_idx] = -np.inf

    # Get top-N movies
    top_indices = np.argsort(
        user_scores
    )[-n:][::-1]

    recommended_movie_ids = [
        index_to_train_movie_id[idx]
        for idx in top_indices
    ]

    return recommended_movie_ids

In [86]:
svd_common_results = []

for i, user_id in enumerate(
    common_evaluation_users,
    start=1
):
    recommendations = generate_svd_recommendations(
        user_id,
        n=10
    )

    relevant_items = test_relevant_by_user_svd.get(
        user_id,
        set()
    )

    if len(recommendations) == 0:
        continue

    svd_common_results.append({
        "userId": user_id,
        "precision@10": precision_at_k(
            recommendations,
            relevant_items,
            10
        ),
        "recall@10": recall_at_k(
            recommendations,
            relevant_items,
            10
        ),
        "ndcg@10": ndcg_at_k(
            recommendations,
            relevant_items,
            10
        )
    })

    if i % 100 == 0:
        print(
            f"Processed {i}/"
            f"{len(common_evaluation_users)} users"
        )

Processed 100/500 users
Processed 200/500 users
Processed 300/500 users
Processed 400/500 users
Processed 500/500 users


In [87]:
svd_common_df = pd.DataFrame(
    svd_common_results
)

print(
    "Successfully evaluated users:",
    len(svd_common_df)
)

print(
    f"SVD Precision@10: "
    f"{svd_common_df['precision@10'].mean():.4f}"
)

print(
    f"SVD Recall@10:    "
    f"{svd_common_df['recall@10'].mean():.4f}"
)

print(
    f"SVD NDCG@10:      "
    f"{svd_common_df['ndcg@10'].mean():.4f}"
)

Successfully evaluated users: 500
SVD Precision@10: 0.0116
SVD Recall@10:    0.1160
SVD NDCG@10:      0.0553


In [88]:
print(
    f"Content Precision@10: "
    f"{evaluation_df['precision@10'].mean():.4f}"
)

print(
    f"Content Recall@10:    "
    f"{evaluation_df['recall@10'].mean():.4f}"
)

print(
    f"Content NDCG@10:      "
    f"{evaluation_df['ndcg@10'].mean():.4f}"
)

Content Precision@10: 0.0036
Content Recall@10:    0.0360
Content NDCG@10:      0.0198


In [89]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "Content-Based",
        "SVD Collaborative",
        "Hybrid"
    ],
    "Content Weight": [
        1.00,
        0.00,
        0.25
    ],
    "Collaborative Weight": [
        0.00,
        1.00,
        0.75
    ],
    "Precision@10": [
        0.0036,
        0.0116,
        0.0114
    ],
    "Recall@10": [
        0.0360,
        0.1160,
        0.1140
    ],
    "NDCG@10": [
        0.0198,
        0.0553,
        0.0573
    ]
})

final_model_comparison

,Model,Content Weight,Collaborative Weight,Precision@10,Recall@10,NDCG@10
0,Content-Based,1.00,0.00,0.0036,0.036,0.0198
1,SVD Collaborative,0.00,1.00,0.0116,0.116,0.0553
2,Hybrid,0.25,0.75,0.0114,0.114,0.0573


In [90]:
final_model_comparison.to_csv(
    "../data/processed/model_comparison.csv",
    index=False
)

print("Final model comparison saved.")

Final model comparison saved.


In [91]:
import sys
sys.path.append("../")

from src.collaborative.matrix_factorization import (
    load_user_item_matrix,
    load_id_mappings,
    fit_svd,
    recommend_for_user
)

print("Collaborative module imported successfully.")

Collaborative module imported successfully.


In [92]:
svd_test, user_factors_test, movie_factors_test = fit_svd(
    train_user_item_matrix,
    n_components=50,
    random_state=42
)

print("SVD fitted successfully.")
print("User factors shape:", user_factors_test.shape)
print("Movie factors shape:", movie_factors_test.shape)

SVD fitted successfully.
User factors shape: (7045, 50)
Movie factors shape: (22187, 50)


In [94]:
test_user_id = train_ratings["userId"].iloc[0]

rated_movie_ids = set(
    train_ratings[
        train_ratings["userId"] == test_user_id
    ]["movieId"]
)

recommendations = recommend_for_user(
    user_id=test_user_id,
    user_id_to_index=train_user_id_to_index,
    movie_id_to_index=train_movie_id_to_index,
    index_to_movie_id=index_to_train_movie_id,
    user_factors=user_factors_test,
    movie_factors=movie_factors_test,
    rated_movie_ids_by_user=rated_movie_ids,
    n=10
)

recommendations

[{'movieId': np.int64(7361),
  'predicted_score': np.float64(1.4360099533181878)},
 {'movieId': np.int64(6874),
  'predicted_score': np.float64(1.2549817904333325)},
 {'movieId': np.int64(778), 'predicted_score': np.float64(1.2177736170886675)},
 {'movieId': np.int64(7153), 'predicted_score': np.float64(1.159136677797361)},
 {'movieId': np.int64(4993),
  'predicted_score': np.float64(1.1326695813393794)},
 {'movieId': np.int64(7438), 'predicted_score': np.float64(1.082594629023905)},
 {'movieId': np.int64(4306), 'predicted_score': np.float64(1.069286944407342)},
 {'movieId': np.int64(4886),
  'predicted_score': np.float64(0.9073833652560571)},
 {'movieId': np.int64(318), 'predicted_score': np.float64(0.8971819840498682)},
 {'movieId': np.int64(4848),
  'predicted_score': np.float64(0.8449144966360995)}]

In [95]:
import sys
sys.path.append("../")

from src.hybrid.blender import (
    normalize_scores,
    blend_scores
)

from src.hybrid.ranker import (
    remove_seen_movies,
    get_top_k,
    rank_hybrid_recommendations
)

print("Hybrid modules imported successfully.")

Hybrid modules imported successfully.


In [96]:
test_content_scores = np.array([
    0.90,
    0.70,
    0.20,
    0.80,
    0.40
])

test_collaborative_scores = np.array([
    0.30,
    0.90,
    0.80,
    0.40,
    0.70
])

test_movies = pd.DataFrame({
    "movieId": [1, 2, 3, 4, 5],
    "title": [
        "Movie A",
        "Movie B",
        "Movie C",
        "Movie D",
        "Movie E"
    ],
    "genres": [
        "Action",
        "Comedy",
        "Drama",
        "Sci-Fi",
        "Thriller"
    ]
})

test_seen_movies = {2}

test_recommendations = rank_hybrid_recommendations(
    content_scores=test_content_scores,
    collaborative_scores=test_collaborative_scores,
    movie_features=test_movies,
    seen_movie_ids=test_seen_movies,
    k=3,
    content_weight=0.25
)

test_recommendations

,movieId,title,genres,score
2,3,Movie C,Drama,0.625000
4,5,Movie E,Thriller,0.571429
3,4,Movie D,Sci-Fi,0.339286


In [101]:
import os

print("Current directory:", os.getcwd())
print("src exists:", os.path.exists("../src"))
print(
    "metrics.py exists:",
    os.path.exists("../src/evaluation/metrics.py")
)

Current directory: c:\Users\amrat\OneDrive\Desktop\MovieRecommendationSystem\notebooks
src exists: True
metrics.py exists: True


In [102]:
from src.evaluation.metrics import (
    precision_at_k,
    recall_at_k,
    ndcg_at_k
)

recommended_items = [1, 2, 3, 4, 5]
relevant_items = {1, 3, 6}

print(
    "Precision@5:",
    precision_at_k(
        recommended_items,
        relevant_items,
        5
    )
)

print(
    "Recall@5:",
    recall_at_k(
        recommended_items,
        relevant_items,
        5
    )
)

print(
    "NDCG@5:",
    ndcg_at_k(
        recommended_items,
        relevant_items,
        5
    )
)

Precision@5: 0.4
Recall@5: 0.6666666666666666
NDCG@5: 0.7039180890341347


In [103]:
from src.evaluation.evaluate import evaluate_recommendations

test_recommendations = {
    1: [1, 2, 3, 4, 5],
    2: [2, 4, 6, 8, 10]
}

test_relevant = {
    1: {1, 3, 6},
    2: {2, 5, 10}
}

evaluation_result = evaluate_recommendations(
    test_recommendations,
    test_relevant,
    k=5
)

evaluation_result

,userId,precision@k,recall@k,ndcg@k
0,1,0.4,0.666667,0.703918
1,2,0.4,0.666667,0.650821


In [104]:
from pathlib import Path

model_dir = Path("../models")

for file in sorted(model_dir.iterdir()):
    print(file.name)

index_to_movie_id.pkl
movie_factors.npy
movie_id_to_index.pkl
svd_model.pkl
tfidf_vectorizer.pkl
user_factors.npy
user_id_to_index.pkl
user_item_matrix.npz
user_rated_movies.pkl
weighted_tfidf_matrix.npz
weighted_tfidf_vectorizer.pkl


In [1]:
import sys

if "../" not in sys.path:
    sys.path.append("../")

In [3]:

from src.model_loader import (
    load_content_models,
    load_collaborative_models
)

print("Model loader imported successfully.")




Model loader imported successfully.


In [4]:
from src.model_loader import (
    load_content_models,
    load_collaborative_models
)

print("Model loader imported successfully.")

Model loader imported successfully.


In [5]:
movie_features_loaded, tfidf_vectorizer_loaded, tfidf_matrix_loaded = (
    load_content_models()
)

collaborative_models_loaded = load_collaborative_models()

print("Movie features:", movie_features_loaded.shape)
print("TF-IDF matrix:", tfidf_matrix_loaded.shape)

print(
    "Users:",
    len(collaborative_models_loaded["user_id_to_index"])
)

print(
    "Movies:",
    len(collaborative_models_loaded["movie_id_to_index"])
)

print(
    "User factors:",
    collaborative_models_loaded["user_factors"].shape
)

print(
    "Movie factors:",
    collaborative_models_loaded["movie_factors"].shape
)

Movie features: (62423, 6)
TF-IDF matrix: (62423, 35768)
Users: 7045
Movies: 22240
User factors: (7045, 50)
Movie factors: (22240, 50)
